### 第 2 周练习 —— 嵌入式工程师「解闷」助手

## 练习目标（理念）

本练习把第 2 周常见能力串起来：**System Prompt + Tools（函数调用）+ SQLite 查价 + DALL·E 配图 + TTS 朗读 + Gradio 聊天 UI**。

- **输入**：和助手闲聊；当你主动要项目点子时，模型会调用工具
- **输出**：文字回复 +（可选）项目/元器件配图 + 语音朗读
- **工具**：
  - `get_component_price`：从本地 SQLite 查元器件价格（不要瞎编价格）
  - `suggest_project`：建议项目时登记名称/简介，供配图使用

## 怎么跑

1. 准备 `.env` 里的 `OPENAI_API_KEY`
2. 从上到下运行单元格（会创建 `components.db`）
3. 最后一格启动 Gradio；在浏览器里聊天


In [ ]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# 从 openai 导入 OpenAI 客户端：云端 Chat / Images / Speech 都走它
from openai import OpenAI
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境变量（Environment Variables）
from dotenv import load_dotenv
# IPython 显示工具：本练习主路径用 Gradio，这里保留以备 Markdown 展示
from IPython.display import Markdown, display, update_display
# 标准库 os：读环境变量，例如 OPENAI_API_KEY
import os
# 标准库 json：解析 tool_call 里的 arguments（模型返回的是 JSON 字符串）
import json
# 标准库 base64：解码 DALL·E 返回的 b64_json 图片数据
import base64
# BytesIO：把字节当成「文件」交给 PIL 打开
from io import BytesIO
# sqlite3：本地组件价格数据库（不依赖外网）
import sqlite3
# PIL.Image：把解码后的图片字节变成可展示的 Image 对象
from PIL import Image
# Gradio：快速搭聊天 + 图片 + 音频的 Web UI
import gradio as gr


In [ ]:
# ========== 初始化：模型名、本地地址、密钥检查 ==========

# 注意：若上面只 import 了 load_dotenv，这里应先调用；原代码未显式调用则保持不动
# OpenAI 云端小模型：便宜、够用，适合带 tools 的对话
MODEL_GPT = 'gpt-4o-mini'
# 本地 Ollama 模型名（本格定义了常量；后面主路径用的是 MODEL_GPT）
MODEL_LLAMA = 'llama3.2'
# Ollama 兼容 OpenAI 的本地 base_url（字符串/URL 不翻译、不改）
OLLAMA_BASE_URL = 'http://localhost:11434/v1'
# 另一处会用 components.db；这里保留原常量 prices.db（逻辑不改）
DB = "prices.db"

# 从环境变量读取 OpenAI API Key（密钥本身不要写进笔记本）
api_key = os.getenv('OPENAI_API_KEY')

# 粗检：是否像 sk-proj- 开头且足够长（仅提示，不阻断后续）
if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    # 给人看的排查文案：影响行为的英文打印保持原样
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")

# 创建默认 OpenAI 客户端（会读环境里的 OPENAI_API_KEY）
openai = OpenAI()


In [ ]:
# ========== 示例问题：嵌入式工程师想找有趣项目 ==========

# 发给模型的内容保持英文（可运行 / 影响回答的字符串不翻译）
# 练习建议：改成你自己的背景或具体芯片平台，再观察工具是否被调用
question = """
Hi there, I'm an embedded software engineer and I'm feeling kinda bored.
Tell me an interesting fact about freertos and exciting projects I can build with it!
I'd be less bored if you could tell me how to build those fun projects!
"""


In [ ]:
# ========== System Prompt：角色 + 何时调用工具 + 禁止瞎编价格 ==========

# system_prompt 整段是「影响模型行为」的英文指令，必须原样保留，不要翻译
# 要点（教学旁注，不改字符串）：
# - 先闲聊，用户主动要项目时再建议
# - 建议项目时必须调用 suggest_project（方便同轮配图）
# - 报价要细；不知道就说不知道，禁止 hallucinate
system_prompt = """you are a helpful assistant that can give a bored embedded software engineer project ideas
                   in a fun way and how to implement them. First, have a conversation with the user and only suggest
                   a project when they ask you to.
                   When you suggest a project to the user, you must call the suggest_project tool with the project
                   name and a short description so an image of the project can be generated for them in the same response.
                   give detailed project ideas in a technical way that such an engineer can implement. also, tell
                   the user how much the project would cost!
                   Give the user a breakdown of the cost. If you don't know the price of a component, 
                   say you don't know and don't guess or hallucinate the price."""


In [ ]:
# ========== SQLite：组件价目表 + 查价工具函数 ==========

# 再次导入 sqlite3（原笔记本如此；保持不删不并）
import sqlite3

# 本练习实际使用的数据库文件名（与上一格 DB="prices.db" 不同，以本格为准）
DB = 'components.db'

# 种子数据：(组件英文名, 美元价格)；名称必须与查询时完全一致才能命中
components_data = [
    ('ESP32-WROOM-32D Development Board', 8.50),
    ('Arduino Nano (Clone)', 4.25),
    ('Raspberry Pi Pico', 4.00),
    ('DHT22 Temperature & Humidity Sensor', 3.75),
    ('HC-SR04 Ultrasonic Distance Sensor', 2.10),
    ('SG90 Micro Servo Motor', 2.50),
    ('16x2 LCD Display with I2C Adapter', 5.50),
    ('TP4056 Li-Po Charging Module', 0.85),
    ('NRF24L01+ Wireless Transceiver', 1.20),
    ('Breadboard (830 Points)', 4.00),
    ('Jumper Wire M-M (40pcs)', 1.50),
    ('10k Ohm Potentiometer', 0.60),
    ('SSD1306 0.96 inch OLED Display', 4.50),
    ('BME280 Pressure/Temp/Hum Sensor', 9.00),
    ('MPU-6050 6-Axis Gyro/Accel', 3.25),
    ('Resistor Kit (600 pcs, Assorted)', 12.00),
    ('Ceramic Capacitor Kit (300 pcs)', 8.50),
    ('Logic Level Converter (4-channel)', 1.10),
    ('Active Buzzer 5V', 0.45),
    ('WS2812B RGB LED Strip (1m/60 LEDs)', 11.00),
    ('Micro SD Card Module', 1.80),
    ('PIR Motion Sensor (HC-SR501)', 1.95),
    ('Shift Register 74HC595', 0.35),
    ('2N2222 NPN Transistor (10pk)', 1.20),
    ('DS3231 RTC Module', 3.80)
]

# 连接（或创建）SQLite 文件；with 结束时自动关闭连接
with sqlite3.connect(DB) as conn:
    # 游标：执行 SQL 语句
    cursor = conn.cursor()
    # 建表：若已存在则跳过（IF NOT EXISTS）
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS components (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            component TEXT NOT NULL,
            price REAL NOT NULL
        )
    """)

    # 查现有行数：避免每次运行都重复插入种子数据
    cursor.execute('SELECT COUNT(*) FROM components')
    count = cursor.fetchone()[0]
    if count == 0:
        # 批量插入种子价目（只有空表时才写）
        cursor.executemany(
            'INSERT INTO components (component, price) VALUES (?, ?)',
            components_data
        )
    # 提交事务，让写入落盘
    conn.commit()

# Tool 实现：按组件全名精确查询价格，返回给人/模型读的字符串
def get_component_price(component_name):
    # flush=True：Jupyter/管道里立刻看到日志，方便确认「工具真的被调用了」
    print(f"DATABASE TOOL CALLED: Getting price for component '{component_name}'", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        # 参数化查询：? 占位，防止拼接 SQL
        cursor.execute('SELECT price FROM components WHERE component = ?', (component_name,))
        result = cursor.fetchone()
        # 命中则格式化美元价；未命中则明确说没有数据（呼应 system：不要猜）
        return f"Price of '{component_name}' is ${result[0]}" if result else f"No price data available for component '{component_name}'"


In [ ]:
# ========== Tools schema：告诉模型「有哪些函数、参数长什么样」 ==========

# OpenAI function calling 的 JSON Schema：name/description/parameters 影响模型是否调用
component_price_function = {
    # 工具名：必须和后面 handle 里判断的字符串一致
    "name": "get_component_price",
    # 描述给模型看：说明何时用、查什么
    "description": "Get the price of the specified electronics component by name.",
    "parameters": {
        "type": "object",
        "properties": {
            "component_name": {
                "type": "string",
                "description": "The name of the electronics component the user wants the price for",
            },
        },
        # 必填参数列表
        "required": ["component_name"],
        # 禁止多余字段，减少胡乱传参
        "additionalProperties": False
    }
}

# 第二个工具：建议项目时登记名称+短描述，供 DALL·E 同轮出图
suggest_project_function = {
    "name": "suggest_project",
    "description": "Call this when you are suggesting a project to the user. Use it as soon as you suggest a project so an image of the project can be shown in the same response.",
    "parameters": {
        "type": "object",
        "properties": {
            "project_name": {
                "type": "string",
                "description": "Short name of the project you are suggesting (e.g. 'Smart Plant Watering System')",
            },
            "short_description": {
                "type": "string",
                "description": "Brief description of the project for image generation (e.g. 'ESP32-based system with soil sensor and pump')",
            },
        },
        "required": ["project_name", "short_description"],
        "additionalProperties": False
    }
}

# 组装成 chat.completions 的 tools= 参数格式：每个元素 type=function + function=schema
tools = [{"type": "function", "function": component_price_function}, {"type": "function", "function": suggest_project_function}]


In [ ]:
# ========== 配图：DALL·E 3 生成组件图 / 项目概念图 ==========

# 按「组件名」生成一张流行艺术风插画（价格查询时的后备配图）
def artist(component_name):
    # 调用 Images API：model / prompt / size 等字符串保持原样
    image_response = openai.images.generate(
        model="dall-e-3",
        prompt=f"A vibrant pop-art illustration highlighting the electronics             component '{component_name}' and its common uses in technology or gadgets.                 Make it visually appealing and professional.",
        size="1024x1024",
        n=1,
        # 要 base64，方便在本地解码成 PIL Image，无需再下载 URL
        response_format="b64_json",
    )
    # 取出第一张图的 b64 字符串
    image_base64 = image_response.data[0].b64_json
    # 解码成原始 PNG/JPEG 字节
    image_data = base64.b64decode(image_base64)
    # 用 BytesIO 包装后交给 PIL 打开
    return Image.open(BytesIO(image_data))


# 按「项目名 + 短描述」生成项目概念图（suggest_project 工具触发时用）
def artist_project(project_name, short_description):
    """为建议的项目生成图像（当法学硕士建议项目时调用）。"""
    image_response = openai.images.generate(
        model="dall-e-3",
        prompt=f"A vibrant, inspiring illustration of an embedded/electronics project: '{project_name}'. "
               f"{short_description}. Show the concept in a clear, technical yet appealing way.",
        size="1024x1024",
        n=1,
        response_format="b64_json",
    )
    image_base64 = image_response.data[0].b64_json
    image_data = base64.b64decode(image_base64)
    return Image.open(BytesIO(image_data))


In [ ]:
# ========== TTS：把助手回复念出来 ==========

# 输入一段文本，返回音频二进制（给 Gradio Audio 组件播放）
def talker(message):
    # OpenAI Speech API：model / voice / input 保持原样
    response = openai.audio.speech.create(
        model="tts-1",
        # 可换成 alloy / coral 等不同音色（英文注释保留原意，旁注说明）
        voice="onyx",  # Try replacing "onyx" with "alloy" or "coral" for different voices
        input=message
    )
    # .content 是音频字节；Gradio 可直接吃这类二进制
    return response.content


In [ ]:
# ========== 处理 tool_calls：执行本地函数，拼回 messages ==========

# 入参 message：模型返回的 assistant message（带 tool_calls）
# 出参：tool 角色回复列表、查过价的组件名列表、本轮建议的项目（若有）
def handle_tool_calls_and_return_components(message):
    # 准备收集：给模型的 tool 回复、组件名、项目建议
    responses = []
    components = []
    suggested_project = None
    # 一轮里可能有多个 tool_call，逐个处理
    for tool_call in message.tool_calls:
        # 分支 A：查组件价格
        if tool_call.function.name == "get_component_price":
            # arguments 是 JSON 字符串 → dict
            arguments = json.loads(tool_call.function.arguments)
            component = arguments.get('component_name')
            # 记下组件名，后面可能用来配「组件图」
            components.append(component)
            # 真正查库
            price_details = get_component_price(component)
            # 按 OpenAI 协议回传：role=tool + 对应 tool_call_id
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
        # 分支 B：登记项目建议（图片稍后在 chat 里生成）
        elif tool_call.function.name == "suggest_project":
            arguments = json.loads(tool_call.function.arguments)
            suggested_project = {
                "project_name": arguments.get("project_name", "Project"),
                "short_description": arguments.get("short_description", ""),
            }
            # 内容字符串会影响模型后续措辞，保持英文原样
            responses.append({
                "role": "tool",
                "content": "Project suggestion recorded; image will be generated for the user.",
                "tool_call_id": tool_call.id
            })
    return responses, components, suggested_project


In [ ]:
# ========== 核心 chat：多轮 tools 循环 + TTS + 配图 ==========

# Gradio 会把 chatbot 历史传进来；本函数返回 (history, voice, image)
def chat(history):
    # 只保留 role/content，去掉 Gradio 可能附带的额外字段
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    # 系统提示 + 对话历史 = 发给模型的完整 messages
    messages = [{"role": "system", "content": system_prompt}] + history
    # 第一次调用：带上 tools，模型可能直接答，也可能 finish_reason=tool_calls
    response = openai.chat.completions.create(model=MODEL_GPT, messages=messages, tools=tools)
    components = []
    suggested_project = None
    image = None

    # 根据要求处理工具调用（get_component_price、suggest_project）
    # 可能连续多轮 tool_calls，直到模型给出最终文本
    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        responses, components, this_suggested = handle_tool_calls_and_return_components(message)
        if this_suggested:
            suggested_project = this_suggested  # keep project suggestion from this turn
        # 把「助手的 tool 请求」原样追加进 messages
        messages.append(message)
        # 再追加我们本地算出来的 tool 结果
        messages.extend(responses)
        # 带着工具结果再问模型
        response = openai.chat.completions.create(model=MODEL_GPT, messages=messages, tools=tools)

    # 最终文本回复（可能为空，用 or "" 兜底）
    reply = response.choices[0].message.content or ""
    # 写回聊天历史，供 Gradio 展示
    history += [{"role": "assistant", "content": reply}]

    # 有文字才做 TTS；否则 voice=None
    voice = talker(reply) if reply else None

    # 当法学硕士建议项目时生成图像（与建议的响应相同）
    if suggested_project:
        image = artist_project(
            suggested_project["project_name"],
            suggested_project["short_description"]
        )
    elif components:
        # 后备：如果仅发生价格查找，则组件图像
        image = artist(components[0])

    return history, voice, image


In [ ]:
# ========== Gradio UI：聊天 + 图片 + 语音 ==========

# 用于将用户消息发布到聊天机器人历史记录中的回调
def put_message_in_chatbot(message, history):
    # 重置输入框（返回 ""），并把用户消息附加到聊天记录
    return "", history + [{"role": "user", "content": message}]

# Gradio UI 定义和事件连接
with gr.Blocks() as ui:
    with gr.Row():
        # type="messages"：历史是 {role, content} 列表，和 OpenAI 风格一致
        chatbot = gr.Chatbot(height=500, type="messages")
        # 右侧展示 DALL·E 图；interactive=False 表示用户不能手动画图
        image_output = gr.Image(height=500, interactive=False)
    with gr.Row():
        # autoplay=True：TTS 返回后自动播放
        audio_output = gr.Audio(autoplay=True)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")

    # 连接文本框提交到聊天更新和 OpenAI + 图像 + 语音管道
    # 第一步：把用户消息塞进 chatbot，并清空输入框
    message.submit(
        put_message_in_chatbot,
        inputs=[message, chatbot],
        outputs=[message, chatbot]
    ).then(
        # 第二步：用更新后的 history 调 chat，刷新 chatbot / 音频 / 图片
        chat,
        inputs=chatbot,
        outputs=[chatbot, audio_output, image_output]
    )

# 通过简单的身份验证在浏览器中启动 UI（用户名/密码字符串保持原样）
ui.launch(inbrowser=True, auth=("denis", "hello"))
